# 🛰️ Sentinel-2 Vegetation Baseline & Anomaly Analysis

**Purpose:** Build a Sentinel-2 climatological baseline (2017–2022), compute anomalies for
analysis years, quantify drought conditions, and classify drought types — mirroring the
MODIS baseline pipeline with full Sentinel-2 specifics.

---

## Workflow Overview
```
1. Setup & AOI loading
2. Build baseline (2017–2022)
   ├── Monthly climatology (mean ± std per calendar month)
   └── Percentile envelopes (10th, 25th, 75th, 90th)
3. Fetch analysis-period data (2023–present)
4. Compute anomalies
   ├── Absolute anomaly  (observed − baseline_mean)
   └── Standardised anomaly / Z-score
5. VCI calculation against the baseline min/max
6. Drought flagging & classification
7. Visualisation
   ├── Long-term baseline ribbon plots
   ├── Anomaly time-series
   ├── Drought heatmaps
   └── Condition summary charts
```

### ⚡ Sentinel-2 vs MODIS — key differences
| Feature | MODIS | Sentinel-2 |
|---|---|---|
| Collection | MOD13Q1 | COPERNICUS/S2_SR_HARMONIZED |
| Resolution | 250 m | 10 m (baseline at 20 m) |
| Available from | 2000 | 2017 |
| Baseline window | 2000–2020 (21 yr) | 2017–2022 (6 yr) |
| Scaling | raw × 0.0001 | divide(10000) in cloud mask |
| Cloud masking | built-in QA | QA60 bit masking |
| NDVI bands | pre-computed | B8 (NIR) / B4 (Red) |
| EVI bands | pre-computed | B8 / B4 / B2 (Blue) |

---

## 1 · Setup

In [1]:
# ── Core imports ─────────────────────────────────────────────────────────────
import ee
import geemap
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings, os, json, zipfile, time
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi'       : 130,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
})

MONTH_ABBR = ['Jan','Feb','Mar','Apr','May','Jun',
              'Jul','Aug','Sep','Oct','Nov','Dec']

print('✅ Imports successful')

✅ Imports successful


In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    # Earth Engine
    'ee_project'        : 'ee-my-ndungu',  # ← change to your project
    's2_collection'     : 'COPERNICUS/S2_SR_HARMONIZED',
    'cloud_pct_max'     : 50,    # pre-filter: max scene cloud %
    'scale_baseline'    : 20,    # metres for baseline (20 m = faster)
    'scale_analysis'    : 10,    # metres for analysis (10 m = full res)

    # Baseline period
    # Sentinel-2 SR available from ~2017; use 2017-2022 as the 6-year baseline
    'baseline_start'    : 2017,
    'baseline_end'      : 2022,

    # Analysis years (post-baseline)
    'analysis_years'    : list(range(2023, 2025)),  # extend as needed

    # Outputs
    'output_dir'        : 's2_baseline_outputs',
}

# Drought thresholds — Z-score based (same as MODIS pipeline)
DROUGHT_THRESHOLDS = {
    'extreme'  : -2.0,
    'severe'   : -1.5,
    'moderate' : -1.0,
    'mild'     : -0.5,
    'normal'   :  0.5,
}

# Drought thresholds — VCI based
VCI_THRESHOLDS = {
    'extreme'  : 10,
    'severe'   : 20,
    'moderate' : 35,
    'mild'     : 50,
    'normal'   : 65,
}

DROUGHT_COLORS = {
    'extreme'      : '#8B0000',
    'severe'       : '#D73027',
    'moderate'     : '#FC8D59',
    'mild'         : '#FEE090',
    'normal'       : '#91BFDB',
    'above_normal' : '#2166AC',
}

# Sentinel-2 wet/dry seasons — East Africa default (customise per region)
# Wet: Mar–May (long rains) and Oct–Dec (short rains)
WET_SEASONS = [(3, 5), (10, 12)]

# Create output subdirectories
for sub in ['baseline', 'anomalies', 'drought', 'trends', 'composites']:
    Path(CONFIG['output_dir'], sub).mkdir(parents=True, exist_ok=True)

print(f"✅ Config ready  |  baseline {CONFIG['baseline_start']}–{CONFIG['baseline_end']}  "
      f"|  analysis years {CONFIG['analysis_years']}")

✅ Config ready  |  baseline 2017–2022  |  analysis years [2023, 2024]


In [5]:
# ── Earth Engine initialisation ───────────────────────────────────────────────
try:
    ee.Initialize(project=CONFIG['ee_project'])
    print(f"✅ Earth Engine initialised  →  project: {CONFIG['ee_project']}")
except Exception:
    print('🔐 Authenticating…')
    ee.Authenticate()
    ee.Initialize(project=CONFIG['ee_project'])
    print('✅ Authenticated & initialised')

✅ Earth Engine initialised  →  project: ee-my-ndungu


## 2 · Load AOI

In [11]:
# ── AOI upload widget ──────────────────────────────────────────────────────────
upload_widget = widgets.FileUpload(
    accept='.zip,.geojson,.json,.shp',
    multiple=False,
    description='📁 Upload AOI',
    style={'description_width': 'initial'}
)
display(upload_widget)
print("⏳ Upload your AOI file above, then run the next cell.")

FileUpload(value=(), accept='.zip,.geojson,.json,.shp', description='📁 Upload AOI')

⏳ Upload your AOI file above, then run the next cell.


In [12]:
# ── Save & load AOI ────────────────────────────────────────────────────────────
def save_upload(widget):
    """Handles both old (dict) and new (tuple) ipywidgets formats."""
    if isinstance(widget.value, dict):
        f = list(widget.value.values())[0]
        name, content = f['metadata']['name'], f['content']
    else:
        f = widget.value[0]
        name, content = f['name'], f['content']
    with open(name, 'wb') as fh:
        fh.write(content)
    return name

def load_aoi(filepath):
    """Load shapefile / GeoJSON / ZIP → GeoDataFrame + EE FeatureCollection."""
    ext = Path(filepath).suffix.lower()
    if ext == '.zip':
        extract_dir = Path('_aoi_tmp')
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(filepath) as z:
            z.extractall(extract_dir)
        shps = list(extract_dir.glob('*.shp'))
        if not shps:
            raise FileNotFoundError('No .shp found inside ZIP')
        gdf = gpd.read_file(shps[0])
    else:
        gdf = gpd.read_file(filepath)
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs('EPSG:4326')
    ee_fc = geemap.geopandas_to_ee(gdf)
    return gdf, ee_fc

if not upload_widget.value:
    raise RuntimeError("⚠️  No file uploaded. Upload an AOI file first.")

aoi_path = save_upload(upload_widget)
aoi_gdf, aoi_ee = load_aoi(aoi_path)
aoi_geom = aoi_ee.geometry()

area_km2 = aoi_gdf.to_crs(epsg=3857).geometry.area.sum() / 1e6
bounds   = aoi_gdf.total_bounds

print(f"✅ AOI loaded   |  features: {len(aoi_gdf)}  |  area: {area_km2:,.1f} km²")
print(f"   Bounds: W {bounds[0]:.3f}  S {bounds[1]:.3f}  E {bounds[2]:.3f}  N {bounds[3]:.3f}")

✅ AOI loaded   |  features: 1  |  area: 2,561.0 km²
   Bounds: W 36.491  S -1.312  E 37.363  N -0.757


In [13]:
# ── Quick AOI preview ──────────────────────────────────────────────────────────
m = geemap.Map()
m.centerObject(aoi_ee, zoom=7)
m.addLayer(aoi_ee, {'color': '#E63946'}, 'AOI')
m

Map(center=[-1.0663333046950283, 36.82274849394715], controls=(WidgetControl(options=['position', 'transparent…

## 3 · Helper Functions (cloud masking, index calculation, reduction)

In [14]:
# ── Sentinel-2 helper functions ───────────────────────────────────────────────

def mask_s2_clouds(image):
    """
    QA60-based cloud + cirrus masking, then scale reflectance to [0, 1].
    Sentinel-2 SR stores reflectance as integers → divide by 10000.
    """
    qa = image.select('QA60')
    cloud_mask  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_mask = qa.bitwiseAnd(1 << 11).eq(0)
    return (image.updateMask(cloud_mask.And(cirrus_mask))
                 .divide(10000)
                 .copyProperties(image, ['system:time_start']))


def add_indices(image):
    """
    Compute NDVI and EVI from Sentinel-2 reflectance bands.
    NDVI = (B8 - B4) / (B8 + B4)
    EVI  = 2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)
    Returns image with NDVI and EVI bands added.
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    evi  = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6.0 * RED - 7.5 * BLUE + 1.0))',
        {'NIR': image.select('B8'),
         'RED': image.select('B4'),
         'BLUE': image.select('B2')}
    ).rename('EVI')
    return image.addBands([ndvi, evi])


def get_s2_collection(start_year, end_year, scale=None):
    """
    Fetch Sentinel-2 SR, apply cloud masking and index calculation.
    Returns a clean collection with NDVI and EVI bands.
    """
    return (ee.ImageCollection(CONFIG['s2_collection'])
              .filterBounds(aoi_geom)
              .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',
                                   CONFIG['cloud_pct_max']))
              .map(mask_s2_clouds)
              .map(add_indices))


def reduce_to_scalar(image, reducer=None):
    """Spatial reduction of an image over the AOI → dict."""
    if reducer is None:
        reducer = ee.Reducer.mean()
    return image.reduceRegion(
        reducer   = reducer,
        geometry  = aoi_geom,
        scale     = CONFIG['scale_baseline'],
        maxPixels = 1e10,
        tileScale = 4,
    ).getInfo()


def monthly_stats_for_year(year, collection):
    """
    Compute monthly mean NDVI & EVI for a single year.
    Returns a list of dicts (one per month that has imagery).
    """
    rows = []
    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        end   = start.advance(1, 'month')
        col   = collection.filterDate(start, end)
        cnt   = col.size().getInfo()
        if cnt == 0:
            continue

        img   = col.mean()
        means = reduce_to_scalar(img, ee.Reducer.mean())
        stds  = reduce_to_scalar(img, ee.Reducer.stdDev())

        rows.append({
            'year'      : year,
            'month'     : m,
            'date'      : pd.Timestamp(year, m, 15),
            'ndvi_mean' : means.get('NDVI'),
            'ndvi_std'  : stds.get('NDVI'),
            'evi_mean'  : means.get('EVI'),
            'evi_std'   : stds.get('EVI'),
            'n_images'  : cnt,
        })
    return rows


print('✅ Sentinel-2 helper functions defined')

✅ Sentinel-2 helper functions defined


## 4 · Build Sentinel-2 Baseline (2017–2022)

> **Runtime note** — this section makes one EE call per year × 12 months at 20 m.
> For a typical national AOI it takes **20–40 minutes** on first run.
> Results are cached to CSV so subsequent runs are instant.

> **Why 2017–2022?**  Sentinel-2B launched in 2017, giving full revisit coverage.
> Six years is short but sufficient for a stable monthly climatology at 10–20 m.

In [ ]:
# ── Fetch or load cached baseline ─────────────────────────────────────────────
BASELINE_CACHE = Path(CONFIG['output_dir']) / 'baseline' / 's2_baseline_monthly_raw.csv'

if BASELINE_CACHE.exists():
    print(f'📦 Loading cached baseline from {BASELINE_CACHE}')
    df_baseline_raw = pd.read_csv(BASELINE_CACHE, parse_dates=['date'])
    print(f'   ✅ Loaded {len(df_baseline_raw)} rows  '
          f'({df_baseline_raw.year.min()}–{df_baseline_raw.year.max()})')
else:
    print(f'📡 Fetching S2 baseline  ({CONFIG["baseline_start"]}–{CONFIG["baseline_end"]})')
    print(f'   Scale: {CONFIG["scale_baseline"]} m  |  cloud filter: <{CONFIG["cloud_pct_max"]}%')
    print('   This will take ~20–40 min for a large AOI — cached afterwards…')

    baseline_coll = get_s2_collection(CONFIG['baseline_start'], CONFIG['baseline_end'])
    total = baseline_coll.size().getInfo()
    print(f'   Total images in baseline collection: {total}\n')

    all_rows = []
    baseline_years = range(CONFIG['baseline_start'], CONFIG['baseline_end'] + 1)

    for i, yr in enumerate(baseline_years):
        print(f'   [{i+1:2d}/{len(baseline_years)}] year {yr} …', end=' ', flush=True)
        t0   = time.time()
        rows = monthly_stats_for_year(yr, baseline_coll)
        all_rows.extend(rows)
        print(f'{len(rows)} months  ({time.time()-t0:.0f}s)')

    df_baseline_raw = pd.DataFrame(all_rows)
    df_baseline_raw.to_csv(BASELINE_CACHE, index=False)
    print(f'\n✅ Baseline saved → {BASELINE_CACHE}')

df_baseline_raw.head()

📡 Fetching S2 baseline  (2017–2022)
   Scale: 20 m  |  cloud filter: <50%
   This will take ~20–40 min for a large AOI — cached afterwards…
   Total images in baseline collection: 835

   [ 1/6] year 2017 … 0 months  (19s)
   [ 2/6] year 2018 … 3 months  (52s)
   [ 3/6] year 2019 … 

In [ ]:
# ── Compute monthly climatology ───────────────────────────────────────────────
# For each calendar month: mean, std, and key percentiles across all baseline years

clim_rows = []
for month in range(1, 13):
    sub = df_baseline_raw[df_baseline_raw.month == month].dropna(
        subset=['ndvi_mean', 'evi_mean'])
    if len(sub) == 0:
        continue

    clim_rows.append({
        'month'          : month,
        'n_years'        : len(sub),

        # NDVI climatology
        'ndvi_clim_mean' : sub.ndvi_mean.mean(),
        'ndvi_clim_std'  : sub.ndvi_mean.std(),
        'ndvi_clim_p10'  : sub.ndvi_mean.quantile(0.10),
        'ndvi_clim_p25'  : sub.ndvi_mean.quantile(0.25),
        'ndvi_clim_p75'  : sub.ndvi_mean.quantile(0.75),
        'ndvi_clim_p90'  : sub.ndvi_mean.quantile(0.90),
        'ndvi_clim_min'  : sub.ndvi_mean.min(),
        'ndvi_clim_max'  : sub.ndvi_mean.max(),

        # EVI climatology
        'evi_clim_mean'  : sub.evi_mean.mean(),
        'evi_clim_std'   : sub.evi_mean.std(),
        'evi_clim_p10'   : sub.evi_mean.quantile(0.10),
        'evi_clim_p25'   : sub.evi_mean.quantile(0.25),
        'evi_clim_p75'   : sub.evi_mean.quantile(0.75),
        'evi_clim_p90'   : sub.evi_mean.quantile(0.90),
        'evi_clim_min'   : sub.evi_mean.min(),
        'evi_clim_max'   : sub.evi_mean.max(),
    })

df_clim = pd.DataFrame(clim_rows)
df_clim.to_csv(Path(CONFIG['output_dir'])/'baseline'/'s2_climatology.csv', index=False)

print('📊 Monthly climatology (S2 baseline):')
print(df_clim[['month','ndvi_clim_mean','ndvi_clim_std',
               'evi_clim_mean','evi_clim_std']].to_string(index=False))

## 5 · Plot the Baseline Climatology

In [ ]:
# ── Baseline climatology ribbon plot ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
fig.suptitle(f'Sentinel-2 Baseline Climatology  '
             f'({CONFIG["baseline_start"]}–{CONFIG["baseline_end"]})',
             fontsize=15, fontweight='bold', y=1.02)

for ax, idx, clr in zip(axes, ['ndvi', 'evi'], ['#2e7d32', '#1565c0']):
    mean = df_clim[f'{idx}_clim_mean'].values
    std  = df_clim[f'{idx}_clim_std'].values
    p10  = df_clim[f'{idx}_clim_p10'].values
    p90  = df_clim[f'{idx}_clim_p90'].values
    p25  = df_clim[f'{idx}_clim_p25'].values
    p75  = df_clim[f'{idx}_clim_p75'].values
    x    = df_clim.month.values

    ax.fill_between(x, p10, p90, alpha=0.15, color=clr, label='10th–90th pct')
    ax.fill_between(x, p25, p75, alpha=0.30, color=clr, label='25th–75th pct')
    ax.fill_between(x, mean - std, mean + std,
                    alpha=0.45, color=clr, label='±1 SD')
    ax.plot(x, mean, 'o-', color=clr, linewidth=2.5,
            markersize=7, label='Climatological mean')

    # Shade wet seasons
    for (ws, we) in WET_SEASONS:
        ax.axvspan(ws - 0.45, we + 0.45, alpha=0.07,
                   color='deepskyblue', label='Wet season' if ws == WET_SEASONS[0][0] else '')

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=10)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel(idx.upper(), fontsize=11)
    ax.set_title(f'{idx.upper()} Monthly Climatology', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
fig.savefig(Path(CONFIG['output_dir'])/'baseline'/'s2_climatology_ribbon.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → baseline/s2_climatology_ribbon.png')

In [ ]:
# ── Annual mean trend across the baseline ─────────────────────────────────────
annual_baseline = (df_baseline_raw
                   .dropna(subset=['ndvi_mean', 'evi_mean'])
                   .groupby('year')[['ndvi_mean', 'evi_mean']]
                   .mean().reset_index())

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle(f'Annual Mean Vegetation Indices — S2 Baseline '
             f'({CONFIG["baseline_start"]}–{CONFIG["baseline_end"]})',
             fontsize=14, fontweight='bold')

for ax, col, clr, label in zip(
        axes,
        ['ndvi_mean', 'evi_mean'],
        ['#2e7d32', '#1565c0'],
        ['NDVI', 'EVI']):

    y = annual_baseline[col].values
    x = annual_baseline['year'].values

    slope, intercept, r, p, _ = scipy_stats.linregress(x, y)
    trend = slope * x + intercept

    ax.bar(x, y, color=clr, alpha=0.55, width=0.7, label=label)
    ax.plot(x, trend, '--', color='black', linewidth=1.8,
            label=f'Trend: {slope:+.5f}/yr  (r={r:.3f}, p={p:.3f})')
    ax.axhline(y.mean(), color=clr, linewidth=1.2, linestyle=':',
               alpha=0.8, label=f'Period mean: {y.mean():.4f}')

    ax.set_ylabel(label, fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))

axes[-1].set_xlabel('Year', fontsize=11)
axes[-1].set_xticks(annual_baseline.year)
axes[-1].set_xticklabels(annual_baseline.year.astype(str), rotation=45, ha='right')

plt.tight_layout()
fig.savefig(Path(CONFIG['output_dir'])/'baseline'/'s2_annual_trend_baseline.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → baseline/s2_annual_trend_baseline.png')

## 6 · Fetch Analysis-Period Data (2023–present)

In [ ]:
# ── Fetch or load cached analysis data ────────────────────────────────────────
ANALYSIS_CACHE = Path(CONFIG['output_dir'])/'anomalies'/'s2_analysis_monthly_raw.csv'

if ANALYSIS_CACHE.exists():
    print(f'📦 Loading cached analysis data from {ANALYSIS_CACHE}')
    df_analysis = pd.read_csv(ANALYSIS_CACHE, parse_dates=['date'])
    print(f'   ✅ Loaded {len(df_analysis)} rows  '
          f'({df_analysis.year.min()}–{df_analysis.year.max()})')
else:
    print(f'📡 Fetching S2 analysis data  ({CONFIG["analysis_years"][0]}–'
          f'{CONFIG["analysis_years"][-1]})')
    print(f'   Scale: {CONFIG["scale_analysis"]} m  |  full Sentinel-2 resolution')

    analysis_coll = get_s2_collection(
        CONFIG['analysis_years'][0], CONFIG['analysis_years'][-1])
    total = analysis_coll.size().getInfo()
    print(f'   Total images: {total}\n')

    all_rows = []
    for i, yr in enumerate(CONFIG['analysis_years']):
        print(f'   [{i+1}/{len(CONFIG["analysis_years"])}] year {yr} …',
              end=' ', flush=True)
        t0   = time.time()
        rows = monthly_stats_for_year(yr, analysis_coll)
        all_rows.extend(rows)
        print(f'{len(rows)} months  ({time.time()-t0:.0f}s)')

    df_analysis = pd.DataFrame(all_rows)
    df_analysis.to_csv(ANALYSIS_CACHE, index=False)
    print(f'\n✅ Saved → {ANALYSIS_CACHE}')

df_analysis.head()

## 7 · Compute Anomalies & VCI

In [ ]:
# ── Merge with climatology and compute anomalies ──────────────────────────────

df_anom = df_analysis.merge(df_clim, on='month', how='left')

for idx in ['ndvi', 'evi']:
    # Absolute anomaly: observed − baseline mean
    df_anom[f'{idx}_anomaly'] = df_anom[f'{idx}_mean'] - df_anom[f'{idx}_clim_mean']

    # Standardised anomaly / Z-score: anomaly ÷ baseline std
    df_anom[f'{idx}_zscore'] = (df_anom[f'{idx}_anomaly']
                                / df_anom[f'{idx}_clim_std'])

    # Percentage anomaly
    df_anom[f'{idx}_pct_anomaly'] = (
        df_anom[f'{idx}_anomaly'] / df_anom[f'{idx}_clim_mean'] * 100)

    # VCI: (observed − baseline_min) / (baseline_max − baseline_min) × 100
    # Uses the climatological min/max from the baseline — NOT within-year
    df_anom[f'{idx}_vci'] = (
        (df_anom[f'{idx}_mean'] - df_anom[f'{idx}_clim_min'])
        / (df_anom[f'{idx}_clim_max'] - df_anom[f'{idx}_clim_min'])
        * 100
    ).clip(0, 100)


# ── Drought classification ─────────────────────────────────────────────────────
def classify_drought_zscore(z):
    if pd.isna(z):                               return 'no_data'
    if z <= DROUGHT_THRESHOLDS['extreme']:        return 'extreme'
    if z <= DROUGHT_THRESHOLDS['severe']:         return 'severe'
    if z <= DROUGHT_THRESHOLDS['moderate']:       return 'moderate'
    if z <= DROUGHT_THRESHOLDS['mild']:           return 'mild'
    if z <=  DROUGHT_THRESHOLDS['normal']:        return 'normal'
    return 'above_normal'

def classify_drought_vci(v):
    if pd.isna(v):                            return 'no_data'
    if v <= VCI_THRESHOLDS['extreme']:         return 'extreme'
    if v <= VCI_THRESHOLDS['severe']:          return 'severe'
    if v <= VCI_THRESHOLDS['moderate']:        return 'moderate'
    if v <= VCI_THRESHOLDS['mild']:            return 'mild'
    if v <= VCI_THRESHOLDS['normal']:          return 'normal'
    return 'above_normal'

df_anom['drought_class_zscore'] = df_anom['ndvi_zscore'].apply(classify_drought_zscore)
df_anom['drought_class_vci']    = df_anom['ndvi_vci'].apply(classify_drought_vci)

out_path = Path(CONFIG['output_dir'])/'anomalies'/'s2_anomalies_full.csv'
df_anom.to_csv(out_path, index=False)

cols = ['year','month','ndvi_mean','ndvi_anomaly','ndvi_zscore',
        'ndvi_vci','drought_class_zscore','drought_class_vci']
print(df_anom[cols].to_string(index=False))
print(f'\n✅ Saved → {out_path}')

## 8 · Visualisation

In [ ]:
# ── Fig 1: Anomaly time-series with baseline ribbon ──────────────────────────

def plot_anomaly_timeseries(df_anom, df_clim, index='ndvi'):
    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 1, height_ratios=[2, 1], hspace=0.08)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    idx_up = index.upper()
    clr    = '#2e7d32' if index == 'ndvi' else '#1565c0'

    dates  = df_anom['date']
    obs    = df_anom[f'{index}_mean']
    clim_m = df_anom[f'{index}_clim_mean']
    clim_s = df_anom[f'{index}_clim_std']
    p10    = df_anom[f'{index}_clim_p10']
    p90    = df_anom[f'{index}_clim_p90']

    # Top panel: observed vs baseline ribbon
    ax1.fill_between(dates, p10, p90,
                     alpha=0.15, color=clr, label='Baseline 10–90th pct')
    ax1.fill_between(dates, clim_m - clim_s, clim_m + clim_s,
                     alpha=0.30, color=clr, label='Baseline ±1 SD')
    ax1.plot(dates, clim_m, '--', color=clr, lw=1.8, alpha=0.9,
             label='Baseline mean')
    ax1.plot(dates, obs, 'o-', color='black', lw=2, ms=5,
             label=f'Observed {idx_up}', zorder=5)

    # Wet season shading
    for yr in df_anom.year.unique():
        for (ws, we) in WET_SEASONS:
            ax1.axvspan(pd.Timestamp(yr, ws, 1),
                        pd.Timestamp(yr, we, 28),
                        alpha=0.05, color='deepskyblue')

    ax1.set_ylabel(idx_up, fontsize=12)
    ax1.set_title(f'Sentinel-2 {idx_up} — Observed vs Baseline  '
                  f'(analysis: {df_anom.year.min()}–{df_anom.year.max()})',
                  fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9, ncol=2, loc='upper right')
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    plt.setp(ax1.get_xticklabels(), visible=False)

    # Bottom panel: Z-score bars
    zscore     = df_anom[f'{index}_zscore']
    bar_colors = [DROUGHT_COLORS[classify_drought_zscore(z)] for z in zscore]

    ax2.bar(dates, zscore, width=25, color=bar_colors, alpha=0.85)
    ax2.axhline(0,  color='black', lw=1.2)
    ax2.axhline(-1, color=DROUGHT_COLORS['moderate'], lw=1, ls='--',
                alpha=0.7, label='Moderate (z=−1)')
    ax2.axhline(-2, color=DROUGHT_COLORS['extreme'],  lw=1, ls='--',
                alpha=0.7, label='Extreme (z=−2)')
    ax2.axhline( 1, color=DROUGHT_COLORS['above_normal'], lw=1, ls='--',
                alpha=0.7, label='Above normal (z=+1)')
    ax2.set_ylabel('Z-score', fontsize=11)
    ax2.set_xlabel('Date',    fontsize=11)
    ax2.legend(fontsize=8, loc='lower right', ncol=3)

    for yr in df_anom.year.unique():
        ax1.axvline(pd.Timestamp(yr, 1, 1), color='grey', lw=0.5, ls=':')
        ax2.axvline(pd.Timestamp(yr, 1, 1), color='grey', lw=0.5, ls=':')

    plt.tight_layout()
    out = Path(CONFIG['output_dir'])/'anomalies'/f's2_{index}_anomaly_timeseries.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {out}')

plot_anomaly_timeseries(df_anom, df_clim, 'ndvi')
plot_anomaly_timeseries(df_anom, df_clim, 'evi')

In [ ]:
# ── Fig 2: VCI time-series with drought threshold bands ──────────────────────
fig, ax = plt.subplots(figsize=(18, 6))

thresholds = [
    (0,  10,  DROUGHT_COLORS['extreme'],     'Extreme drought'),
    (10, 20,  DROUGHT_COLORS['severe'],      'Severe drought'),
    (20, 35,  DROUGHT_COLORS['moderate'],    'Moderate drought'),
    (35, 50,  DROUGHT_COLORS['mild'],        'Mild drought'),
    (50, 65,  DROUGHT_COLORS['normal'],      'Normal'),
    (65, 100, DROUGHT_COLORS['above_normal'],'Above normal'),
]
for lo, hi, clr, lbl in thresholds:
    ax.axhspan(lo, hi, alpha=0.12, color=clr, label=lbl)

ax.plot(df_anom['date'], df_anom['ndvi_vci'],
        'o-', color='black', lw=2, ms=5, zorder=5, label='NDVI-VCI (vs baseline)')

scatter_colors = [DROUGHT_COLORS[c] for c in df_anom['drought_class_vci']]
ax.scatter(df_anom['date'], df_anom['ndvi_vci'],
           c=scatter_colors, s=60, zorder=6,
           edgecolors='white', linewidths=0.5)

# Wet season shading
for yr in df_anom.year.unique():
    for (ws, we) in WET_SEASONS:
        ax.axvspan(pd.Timestamp(yr, ws, 1),
                   pd.Timestamp(yr, we, 28),
                   alpha=0.05, color='deepskyblue')

for yr in df_anom.year.unique():
    ax.axvline(pd.Timestamp(yr, 1, 1), color='grey', lw=0.5, ls=':')

ax.set_ylim(0, 100)
ax.set_ylabel('VCI  (0 = worst, 100 = best)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_title('Sentinel-2 Vegetation Condition Index (VCI)  —  '
             'Against S2 Baseline (2017–2022)',
             fontsize=13, fontweight='bold')

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, fontsize=9, ncol=4,
          loc='lower right', framealpha=0.85)

plt.tight_layout()
out = Path(CONFIG['output_dir'])/'drought'/'s2_vci_timeseries.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {out}')

In [ ]:
# ── Fig 3: Drought classification heatmap (year × month) ─────────────────────
CLASS_ORDER = ['extreme','severe','moderate','mild','normal','above_normal']
CLASS_INT   = {c: i for i, c in enumerate(CLASS_ORDER)}
CMAP        = mcolors.ListedColormap([DROUGHT_COLORS[c] for c in CLASS_ORDER])
NORM        = mcolors.BoundaryNorm(range(len(CLASS_ORDER) + 1), CMAP.N)

for method, col in [('Z-score', 'drought_class_zscore'),
                    ('VCI',     'drought_class_vci')]:
    pivot = (df_anom
             .pivot_table(index='year', columns='month',
                          values=col, aggfunc='first')
             .apply(lambda s: s.map(lambda x: CLASS_INT.get(x, np.nan))))

    fig, ax = plt.subplots(
        figsize=(14, max(4, len(pivot) * 0.7 + 1)))

    ax.imshow(pivot.values, cmap=CMAP, norm=NORM,
              aspect='auto', interpolation='nearest')

    SHORT = {'extreme':'EX','severe':'SV','moderate':'MD',
             'mild':'ML','normal':'NL','above_normal':'AN'}
    for i, yr in enumerate(pivot.index):
        for j, mo in enumerate(pivot.columns):
            val = pivot.loc[yr, mo]
            if not (isinstance(val, float) and np.isnan(val)):
                cls   = CLASS_ORDER[int(val)]
                short = SHORT[cls]
                ax.text(j, i, short, ha='center', va='center',
                        fontsize=8, fontweight='bold',
                        color='white' if int(val) <= 1 else 'black')

    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_ABBR, fontsize=10)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index.astype(str), fontsize=10)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel('Year',  fontsize=11)
    ax.set_title(f'Sentinel-2 Drought Classification  ({method})  —  '
                 f'{df_anom.year.min()}–{df_anom.year.max()}',
                 fontsize=13, fontweight='bold')

    patches = [mpatches.Patch(
                   color=DROUGHT_COLORS[c],
                   label=c.replace('_',' ').title())
               for c in CLASS_ORDER]
    ax.legend(handles=patches, fontsize=9,
              bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.tight_layout()
    fname = method.lower().replace('-','').replace(' ','_')
    out   = Path(CONFIG['output_dir'])/'drought'/f's2_drought_heatmap_{fname}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {out}')

In [ ]:
# ── Fig 4: Percentage anomaly bars by year & month ───────────────────────────
fig, axes = plt.subplots(len(CONFIG['analysis_years']), 1,
                         figsize=(16, 4 * len(CONFIG['analysis_years'])),
                         sharex=True)
if len(CONFIG['analysis_years']) == 1:
    axes = [axes]

fig.suptitle('Sentinel-2 NDVI Percentage Anomaly  '
             '(% departure from S2 baseline)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, yr in zip(axes, CONFIG['analysis_years']):
    sub  = df_anom[df_anom.year == yr].copy()
    vals = sub['ndvi_pct_anomaly'].values
    mths = sub['month'].values
    clrs = ['#D73027' if v < 0 else '#2166AC' for v in vals]

    # Wet season shading
    for (ws, we) in WET_SEASONS:
        ax.axvspan(ws - 0.5, we + 0.5,
                   alpha=0.07, color='deepskyblue')

    bars = ax.bar(mths, vals, color=clrs, alpha=0.80,
                  width=0.7, edgecolor='white')
    ax.axhline(0,   color='black', lw=1.2)
    ax.axhline(-10, color='orange',    lw=0.8, ls='--', alpha=0.6)
    ax.axhline( 10, color='steelblue', lw=0.8, ls='--', alpha=0.6)

    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2,
                    val + (1 if val >= 0 else -1),
                    f'{val:+.1f}%',
                    ha='center',
                    va='bottom' if val >= 0 else 'top',
                    fontsize=8.5)

    ax.set_ylabel('% Anomaly', fontsize=10)
    ax.set_title(str(yr), fontsize=11, loc='left', fontweight='bold')
    ax.set_xlim(0.3, 12.7)

axes[-1].set_xticks(range(1, 13))
axes[-1].set_xticklabels(MONTH_ABBR, fontsize=10)

plt.tight_layout()
out = Path(CONFIG['output_dir'])/'anomalies'/'s2_ndvi_pct_anomaly_bars.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {out}')

In [ ]:
# ── Fig 5: Drought frequency & severity summary ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Sentinel-2 Drought Summary  —  Analysis Period',
             fontsize=14, fontweight='bold')

# Panel A: Frequency of each class
ax = axes[0]
freq = (df_anom['drought_class_zscore']
        .value_counts()
        .reindex(CLASS_ORDER, fill_value=0))
bars = ax.barh(freq.index, freq.values,
               color=[DROUGHT_COLORS[c] for c in freq.index],
               edgecolor='white', alpha=0.85)
for bar, val in zip(bars, freq.values):
    ax.text(bar.get_width() + 0.3,
            bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
ax.set_xlabel('Number of months', fontsize=11)
ax.set_title('(A) Class frequency (Z-score)', fontsize=11, fontweight='bold')
ax.set_yticklabels([c.replace('_',' ').title() for c in freq.index], fontsize=10)

# Panel B: Monthly drought frequency
ax = axes[1]
drought_months = df_anom[df_anom['drought_class_zscore'].isin(
    ['extreme','severe','moderate','mild'])]
month_freq = (drought_months.groupby('month').size()
              .reindex(range(1, 13), fill_value=0))
clrs_bar = ['#D73027' if v >= 2 else '#FC8D59' if v == 1 else '#91BFDB'
            for v in month_freq.values]
ax.bar(range(1, 13), month_freq.values,
       color=clrs_bar, alpha=0.85, edgecolor='white')
# Overlay wet season shading
for (ws, we) in WET_SEASONS:
    ax.axvspan(ws - 0.5, we + 0.5, alpha=0.08, color='deepskyblue',
               label='Wet season' if ws == WET_SEASONS[0][0] else '')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_ABBR, fontsize=9)
ax.set_ylabel('Number of drought months', fontsize=11)
ax.set_title('(B) Drought months per calendar month', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Panel C: Annual drought severity
ax = axes[2]
annual_severity = (df_anom.groupby('year')['ndvi_zscore']
                   .apply(lambda x: x[x < 0].abs().mean())
                   .fillna(0))
bar_clrs = ['#8B0000' if v > 1.5 else '#D73027' if v > 1.0
            else '#FC8D59' if v > 0.5 else '#91BFDB'
            for v in annual_severity.values]
ax.bar(annual_severity.index, annual_severity.values,
       color=bar_clrs, alpha=0.85, edgecolor='white')
ax.axhline(1.0, color='red',     lw=1.2, ls='--', label='Moderate threshold')
ax.axhline(1.5, color='darkred', lw=1.2, ls='--', label='Severe threshold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Mean |Z-score| (drought months only)', fontsize=10)
ax.set_title('(C) Annual drought severity', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xticks(annual_severity.index)
ax.set_xticklabels(annual_severity.index.astype(str), rotation=45, ha='right')

plt.tight_layout()
out = Path(CONFIG['output_dir'])/'drought'/'s2_drought_summary.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {out}')

In [ ]:
# ── Fig 6: Full long-term view  (baseline + analysis combined) ───────────────
df_full = pd.concat([
    df_baseline_raw.assign(period='baseline'),
    df_analysis.assign(period='analysis')
], ignore_index=True).sort_values('date').reset_index(drop=True)

ann = (df_full.dropna(subset=['ndvi_mean','evi_mean'])
       .groupby(['year','period'])[['ndvi_mean','evi_mean']]
       .mean().reset_index())

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'Sentinel-2 Long-term Vegetation Record  '
             f'({CONFIG["baseline_start"]}–{df_full.year.max()})',
             fontsize=14, fontweight='bold')

for ax, col, clr, label in zip(
        axes,
        ['ndvi_mean','evi_mean'],
        ['#2e7d32','#1565c0'],
        ['NDVI','EVI']):

    base_sub = ann[ann.period == 'baseline']
    anal_sub = ann[ann.period == 'analysis']

    ax.bar(base_sub.year, base_sub[col], width=0.8,
           color=clr, alpha=0.50,
           label=f'{label} baseline ({CONFIG["baseline_start"]}–{CONFIG["baseline_end"]})')
    ax.bar(anal_sub.year, anal_sub[col], width=0.8,
           color='#d32f2f', alpha=0.65,
           label=f'{label} analysis')

    bline = base_sub[col].mean()
    ax.axhline(bline, color=clr, lw=1.8, ls='--',
               label=f'Baseline mean: {bline:.4f}')

    ax.axvspan(CONFIG['baseline_start'] - 0.5,
               CONFIG['baseline_end']   + 0.5,
               alpha=0.05, color='grey', label='Baseline period')

    x_all = ann.year.values
    y_all = ann[col].values
    mask  = ~np.isnan(y_all)
    slope, intercept, r, p, _ = scipy_stats.linregress(
        x_all[mask], y_all[mask])
    ax.plot(x_all[mask], slope * x_all[mask] + intercept,
            '-', color='black', lw=1.5,
            label=f'Overall trend {slope:+.5f}/yr (p={p:.3f})')

    ax.set_ylabel(label, fontsize=12)
    ax.legend(fontsize=9, ncol=2, loc='upper right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))

axes[-1].set_xlabel('Year', fontsize=12)
axes[-1].set_xticks(ann.year.unique())
axes[-1].set_xticklabels(ann.year.unique().astype(str),
                         rotation=45, ha='right')

plt.tight_layout()
out = Path(CONFIG['output_dir'])/'trends'/'s2_longterm_combined.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {out}')

In [ ]:
# ── Fig 7: Seasonal NDVI anomaly breakdown per analysis year ─────────────────
# East Africa seasons: MAM = long rains, OND = short rains
seasons = {
    'DJF': [12, 1, 2],
    'MAM': [3, 4, 5],   # Long rains
    'JJA': [6, 7, 8],
    'OND': [9, 10, 11], # Short rains (replaces SON to match Oct–Dec)
}

def season_of(month):
    for s, months in seasons.items():
        if month in months:
            return s
    return 'Unknown'

df_anom['season'] = df_anom['month'].apply(season_of)
seas_anom = (df_anom
             .groupby(['year','season'])
             [['ndvi_anomaly','ndvi_zscore','ndvi_vci']]
             .mean().reset_index())

fig, ax = plt.subplots(figsize=(15, 6))

season_offsets = {'DJF': -0.3, 'MAM': -0.1, 'JJA': 0.1, 'OND': 0.3}
season_colors  = {'DJF': '#2196F3', 'MAM': '#4CAF50',
                  'JJA': '#FF9800', 'OND': '#9C27B0'}
bar_w = 0.18

for sname, offset in season_offsets.items():
    sub  = seas_anom[seas_anom.season == sname]
    xpos = sub.year.values + offset
    vals = sub.ndvi_anomaly.values
    ax.bar(xpos, vals, width=bar_w,
           color=season_colors[sname], alpha=0.80,
           label=sname, edgecolor='white')

ax.axhline(0, color='black', lw=1.2)
ax.set_xticks(CONFIG['analysis_years'])
ax.set_xticklabels([str(y) for y in CONFIG['analysis_years']], fontsize=11)
ax.set_ylabel('NDVI Anomaly (absolute)', fontsize=12)
ax.set_title('Sentinel-2 Seasonal NDVI Anomalies vs Baseline (2017–2022)',
             fontsize=13, fontweight='bold')
ax.legend(title='Season', fontsize=10, title_fontsize=10)

plt.tight_layout()
out = Path(CONFIG['output_dir'])/'anomalies'/'s2_seasonal_anomaly.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {out}')

## 9 · Drought Summary Table

In [ ]:
# ── Drought event catalogue ───────────────────────────────────────────────────
from IPython.display import HTML

drought_events = df_anom[
    df_anom['drought_class_zscore'].isin(['extreme','severe','moderate'])
].copy()

drought_events['month_name'] = drought_events['month'].apply(
    lambda m: MONTH_ABBR[m - 1])

display_cols = {
    'year'               : 'Year',
    'month_name'         : 'Month',
    'ndvi_mean'          : 'NDVI',
    'ndvi_clim_mean'     : 'Baseline mean',
    'ndvi_anomaly'       : 'Anomaly',
    'ndvi_zscore'        : 'Z-score',
    'ndvi_vci'           : 'VCI',
    'ndvi_pct_anomaly'   : '% Anomaly',
    'drought_class_zscore': 'Class (Z)',
    'drought_class_vci'  : 'Class (VCI)',
}

table = drought_events[list(display_cols)].rename(columns=display_cols).copy()
for col in ['NDVI','Baseline mean','Anomaly','Z-score','VCI','% Anomaly']:
    if col in table.columns:
        table[col] = table[col].round(4)

def row_style(row):
    c  = row['Class (Z)']
    bg = {'extreme':'#ffcccc','severe':'#ffddb3',
          'moderate':'#fff5b3','mild':'#e8f5e9'}.get(c, '')
    return [f'background-color: {bg}'] * len(row)

styled = (table.style
               .apply(row_style, axis=1)
               .format({'NDVI':'{:.4f}','Baseline mean':'{:.4f}',
                        'Anomaly':'{:+.4f}','Z-score':'{:+.3f}',
                        'VCI':'{:.1f}','% Anomaly':'{:+.1f}%'}))

if len(table) == 0:
    print('✅ No drought events detected in the analysis period.')
else:
    print(f'⚠️  {len(table)} drought months detected:')
    display(styled)

csv_path = Path(CONFIG['output_dir'])/'drought'/'s2_drought_event_catalogue.csv'
table.to_csv(csv_path, index=False)
print(f'\n💾 Saved → {csv_path}')

In [ ]:
# ── Final statistics summary ─────────────────────────────────────────────────
print('\n' + '═' * 65)
print('  SENTINEL-2 ANALYSIS SUMMARY')
print('═' * 65)
print(f"  Sensor            : Sentinel-2 SR Harmonised (10 m)")
print(f"  S2 collection     : {CONFIG['s2_collection']}")
print(f"  Baseline period   : {CONFIG['baseline_start']}–{CONFIG['baseline_end']} "
      f"({CONFIG['baseline_end'] - CONFIG['baseline_start'] + 1} years)")
print(f"  Analysis period   : {CONFIG['analysis_years'][0]}–{CONFIG['analysis_years'][-1]}")
if aoi_gdf is not None:
    area_km2 = aoi_gdf.to_crs(epsg=3857).geometry.area.sum() / 1e6
    print(f'  AOI area          : {area_km2:,.1f} km²')
print(f"  Baseline scale    : {CONFIG['scale_baseline']} m")
print(f"  Analysis scale    : {CONFIG['scale_analysis']} m")
print(f"  Cloud filter      : <{CONFIG['cloud_pct_max']}% CLOUDY_PIXEL_PERCENTAGE")

print('\n  BASELINE CLIMATOLOGY (NDVI)')
print(f"  {'Month':<6} {'Mean':>8} {'Std':>8} {'P10':>8} {'P90':>8}")
print('  ' + '-'*38)
for _, row in df_clim.iterrows():
    print(f"  {MONTH_ABBR[int(row.month)-1]:<6} "
          f"{row.ndvi_clim_mean:>8.4f} "
          f"{row.ndvi_clim_std:>8.4f} "
          f"{row.ndvi_clim_p10:>8.4f} "
          f"{row.ndvi_clim_p90:>8.4f}")

print('\n  DROUGHT MONTHS BY CLASS (Z-score, analysis period)')
for cls in CLASS_ORDER:
    n   = (df_anom['drought_class_zscore'] == cls).sum()
    bar = '█' * n
    print(f'  {cls.replace("_"," ").title():<15} {n:>3}  {bar}')

total_months   = len(df_anom)
drought_months = (df_anom['drought_class_zscore']
                  .isin(['extreme','severe','moderate','mild'])
                  .sum())
print(f'\n  Total months analysed : {total_months}')
print(f'  Drought months (mild+): {drought_months}  '
      f'({100*drought_months/total_months:.1f}%)')
print('═' * 65)

In [ ]:
# ── List all saved outputs ────────────────────────────────────────────────────
print('📁 All saved outputs:')
for f in sorted(Path(CONFIG['output_dir']).rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'   {str(f):<65}  {size:>7.1f} KB')